In [ ]:
# we are following Karpathy's straightforward nanogpt

In [21]:
# Source - https://stackoverflow.com/a/10472712
# Posted by Andrew_1510, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-12, License - CC BY-SA 4.0

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from gpt2 import GPT, gpt2_generate

In [23]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
max_length = 30

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
model = GPT.from_pretrained("gpt2")
model.to(device)

loading weights from pretrained gpt: gpt2


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [24]:
# using gpt2_generate

tokens = tokenizer.encode("The CEO is here and he wants to")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(5, 1)
x = tokens.to(device)
log_probs, generated_tokens, full_sequence = gpt2_generate(model, x, gen_len=10, max_seq_length=1024)
for i in range(5):
    print(tokenizer.decode(generated_tokens[i, :max_length].tolist()))
print("_"*50)
for i in range(5):
    print(tokenizer.decode(full_sequence[i, :max_length].tolist()))

 ensure that the Secret World is as safe as Religion
 hear from you, well him. But something that
 help you learn and comes to work every day.
 discuss those things, but there's problems that all
 just own these things, which you say he is
__________________________________________________
The CEO is here and he wants to ensure that the Secret World is as safe as Religion
The CEO is here and he wants to hear from you, well him. But something that
The CEO is here and he wants to help you learn and comes to work every day.
The CEO is here and he wants to discuss those things, but there's problems that all
The CEO is here and he wants to just own these things, which you say he is


In [25]:
# using gpt2_generate with the interp args

tokens = tokenizer.encode("The CEO is here and")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(7, 1)
x = tokens.to(device)
log_probs, generated_tokens, full_sequence, pooled_neurons = gpt2_generate(model, x, gen_len=2, max_seq_length=1024, target_layer=8, return_neurons=True, pooling="mean")
for i in range(7):
    print(tokenizer.decode(generated_tokens[i, :max_length].tolist()))
print("_"*50)
for i in range(7):
    print(tokenizer.decode(full_sequence[i, :max_length].tolist()))
    
print(f"pooled_neurons: {pooled_neurons.shape}")

 the CEO
 asked me
 is here
 saying he
 looking at
 he'll
 he praised
__________________________________________________
The CEO is here and the CEO
The CEO is here and asked me
The CEO is here and is here
The CEO is here and saying he
The CEO is here and looking at
The CEO is here and he'll
The CEO is here and he praised
pooled_neurons: torch.Size([7, 3072])


In [41]:
def scaled_input(emb, batch_size, num_batch):
    # emb is the ffn_activation/ffn_neurons
    baseline = torch.zeros_like(emb)

    num_points = batch_size * num_batch
    step = (emb - baseline) / num_points
    res_list = [torch.add(baseline, step * i) for i in range(num_points)]
    
    print(f"res_list: {len(res_list)}")
    print(f"vector shape: {res_list[0].shape}")
    
    res = torch.cat(res_list, dim=0)
    return res, step[0]

In [39]:
scaled_neurons, neuron_step = scaled_input(pooled_neurons, 16, 10)

res_list: 160
vector shape: torch.Size([7, 3072])


In [40]:
scaled_neurons.shape

torch.Size([1120, 3072])

In [13]:
logits, _, mlp_inter_neurons = model(x, target_layer=8, return_neurons=True)
logits.shape, mlp_inter_neurons.shape

(torch.Size([5, 8, 50257]), torch.Size([5, 8, 3072]))

In [55]:
tokens = tokenizer.tokenize("The nurse is here and she ")
print(tokens)
print(tokenizer.convert_tokens_to_ids(tokens))

tokenizer.convert_tokens_to_ids(tokenizer.eos_token)

['The', 'Ġnurse', 'Ġis', 'Ġhere', 'Ġand', 'Ġshe', 'Ġ']
[464, 15849, 318, 994, 290, 673, 220]


50256

In [57]:
tokenizer.tokenize("indian")

['ind', 'ian']

In [81]:
tokens = tokenizer.encode("I am GPT and I ")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(1, 1)
x = tokens.to(device)

logits, _ = model(x) # (B, T, vocab_size)
logits = logits[:, -1, :] # (B, vocab_size)
# get the probabilities
probs = torch.nn.functional.softmax(logits, dim=-1)
# do top-k sampling of 50 (huggingface pipeline default)
# topk_probs here becomes (5, 50), topk_indices is (5, 50)
topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
# select a token from the top-k probabilities
# note: multinomial does not demand the input to sum to 1
ix = torch.multinomial(topk_probs, 1) # (B, 1)
# gather the corresponding indices
xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
# append to the sequence
x = torch.cat((x, xcol), dim=1)
tokens = x[0, :max_length].tolist()
decoded = tokenizer.decode(tokens)

print(tokens)
print(f"'{decoded}'")
print(tokenizer.encode(decoded))

[40, 716, 402, 11571, 290, 314, 220, 425]
'I am GPT and I ive'
[40, 716, 402, 11571, 290, 314, 220, 425]


In [52]:
tokenizer.encode(" she")

[673]

In [10]:
while x.size(1) < max_length:
    # forward the model to get the logits
    with torch.no_grad():
        logits, _, mlp_inter_neurons = model(x, target_layer=8, return_neurons=True) # (B, T, vocab_size)
        # take the logits at the last position
        logits = logits[:, -1, :] # (B, vocab_size)
        # get the probabilities
        probs = torch.nn.functional.softmax(logits, dim=-1)
        # do top-k sampling of 50 (huggingface pipeline default)
        # topk_probs here becomes (5, 50), topk_indices is (5, 50)
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
        # select a token from the top-k probabilities
        # note: multinomial does not demand the input to sum to 1
        ix = torch.multinomial(topk_probs, 1) # (B, 1)
        # gather the corresponding indices
        xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
        # append to the sequence
        x = torch.cat((x, xcol), dim=1)

print(f"mlp_neurons: {mlp_inter_neurons.shape}")

# print the generated text
for i in range(5):
    tokens = x[i, :max_length].tolist()
    decoded = tokenizer.decode(tokens)
    print(">", decoded)

mlp_neurons: torch.Size([5, 29, 3072])
> The nurse is here and she ers and she does the right things.

She's very open, friendly, intelligent and caring. Her body
> The nurse is here and she  told me they'd get a better understanding of everything, including how to get rid of the dog, how to
> The nurse is here and she __________ is telling us that this time, he has broken the rules that should be followed. It is time for
> The nurse is here and she ers pretty sure you don't want it.

"Why the hell don't I tell her about this?"
> The nurse is here and she ive spoken to my girlfriend in person so I can keep her informed.

Can you help them be safe?
